#### 3.1. Verifying libraries

In [3]:
from importlib.metadata import version

used_libraries = [
    'reasoning_from_scratch',
    'torch',
    'sympy',
    'tokenizers'
]

for lib in used_libraries:
    try:
        v = version(lib)
        print(f'{lib}: {v}')
    except Exception as e:
        print(f'{lib}: Not found')

reasoning_from_scratch: Not found
torch: 2.4.1+cu124
sympy: 1.14.0
tokenizers: 0.23.3


#### 3.2. Loading a pre-trained model to generate text

In [8]:
from pathlib import Path
import torch

from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

def load_model_and_tokenizer(
        which_model, device, use_compile, local_dir="qwen3"
):
    if which_model == "base":
        download_qwen3_small(
            kind="base", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-base.json"
        model_path = Path(local_dir) / "qwen3-0.6B-base.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

    elif which_model == "reasoning":
        download_qwen3_small(
            kind="reasoning", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
        model_path = Path(local_dir) / "qwen3-0.6B-reasoning.pth"
        tokenizer = Qwen3Tokenizer(
            tokenizer_file_path=tokenizer_path,
            apply_chat_template=True,
            add_generation_prompts=True,
            add_thinking=True
        )

    else:
        raise ValueError("which_model must be 'base' or 'reasoning'")

    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))
    model = model.to(device)

    if use_compile:
        torch.compile(model, mode="reduce-overhead", fullgraph=True)

    return model, tokenizer

In [10]:
from reasoning_from_scratch.ch02 import (
    get_device
)

WHICH_MODEL = "base"
device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model=WHICH_MODEL,
    device=device,
    use_compile=False
)

print(device)

Using NVIDIA CUDA GPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


C:\Users\Eliey\AppData\Local\Temp\ipykernel_21868\3447637962.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


cuda


In [33]:
import torch

from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache
)

prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
).unsqueeze(0)

all_token_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,
    eos_token_id=tokenizer.eos_token_id
):
    # Extract scalar token ID (handles both 1D and 2D tensor outputs)
    token_id = token.squeeze().item()
    
    # Store plain integer ID for final decoding
    all_token_ids.append(token_id)
    
    # Stream decode single token to console
    decoded_text = tokenizer.decode([token_id])
    print(decoded_text, end="", flush=True)

# Decode all generated token IDs back to text cleanly
all_tokens = tokenizer.decode(all_token_ids)

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

In [34]:
from IPython.display import Math, Latex, display

display(Latex(all_tokens))

display(Math(r" \boxed{\dfrac{14}{3}}"))

<IPython.core.display.Latex object>

<IPython.core.display.Math object>

#### 3.3. Implementing a wrapper for easier text generation

In [36]:
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache
)

def generate_text_stream_concat(
    model, tokenizer, prompt, device, max_new_tokens, verbose=False
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device
    ).unsqueeze(0)

    generated_ids = []

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze().item()
        generated_ids.append(token_id)

        if verbose:
            decoded_text = tokenizer.decode([token_id])
            print(decoded_text, end="", flush=True)
    
    return tokenizer.decode(generated_ids)

skip_portion = True

if not skip_portion:
    generated_text = generate_text_stream_concat(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=2048,
        verbose=True
    )

#### 3.4. Extracting the final answer box

In [49]:
model_answer = (
r"""... some explanation...
**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]
""")

In [52]:
def get_last_boxed(text):
    # find the last occurence of `\boxed`
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None

    # get position after `\boxed`
    current_idx = boxed_start_idx + len(r"\boxed")

    # skip any whitespace after `\boxed`
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # expect an opening braces `{`
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    # parse the content inside the braces
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx] 
        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1
        current_idx += 1 

    # account for unbalanced braces
    if brace_depth != 0:
        return None

    # extract content inside the outmost braces
    return text[content_start_idx:current_idx-1]

answer = get_last_boxed(model_answer)
print(answer)

\dfrac{14}{3}


#### 3.5. Handling answers without boxes

In [59]:
import re

RE_NUMBER = re.compile(
    r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
)

def extract_final_candidate(text, fallback="number_then_full"):
    # default return value
    result = ""

    if text:
        # prefer the last boxed expression
        boxed = get_last_boxed(text.strip())
        if boxed:
            result = boxed.strip().strip("$ ")

        # if no boxed expression, fall back to number extraction
        elif fallback in ["number_then_full", "number_only"]:
            m = RE_NUMBER.findall(text.strip())
            if m:
                result = m[-1].strip()
            elif fallback == "number_then_full":
                # else return full text if no number found
                result = text

    return result

print(extract_final_candidate(model_answer))
print(extract_final_candidate(r"\boxed{ 14/3. }"))
print(extract_final_candidate("abc < > 14/3 .abc"))
print(extract_final_candidate("Text without numbers"))

\dfrac{14}{3}
14/3.
14/3
Text without numbers


#### 3.6. Normalizing the extracted answer

In [60]:
LATEX_FIXES = [  # Latex formatting to be replaced
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]

RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")  # strip chat special tokens like <|assistant|>

def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()
    SUPERSCRIPT_MAP = {
        "⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",
        "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9",
        "⁺": "+", "⁻": "-", "⁽": "(", "⁾": ")",
    }

    # Strip leading multiple-choice labels
    # E.g., like "c. 3" -> 3, or "b: 2" -> 2
    match = re.match(r"^[A-Za-z]\s*[.:]\s*(.+)$", text)
    if match:
        text = match.group(1)
        
    # Remove angle-degree markers
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)   # ^{\circ}
    text = re.sub(r"\^\s*\\circ", "", text)             # ^\circ
    text = text.replace("°", "")                        # Unicode degree

    # unwrap \text{...} if the whole string is wrapped
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    # strip inline/display math wrappers \( \) \[ \]
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # light LaTeX canonicalization
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    def convert_superscripts(s, base=None):
        converted = "".join(
            SUPERSCRIPT_MAP[ch] if ch in SUPERSCRIPT_MAP else ch
            for ch in s
        )
        if base is None:
            return converted
        return f"{base}**{converted}"

    # convert unicode superscripts into exponent form (e.g., 2² -> 2**2)m
    text = re.sub(
        r"([0-9A-Za-z\)\]\}])([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)",
        lambda m: convert_superscripts(m.group(2), base=m.group(1)),
        text,
    )
    text = convert_superscripts(text)
    
    # numbers/roots
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # fractions
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # exponent and mixed numbers
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # 1,234 -> 1234
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    return text.replace("{", "").replace("}", "").strip().lower()

print(normalize_text(extract_final_candidate(model_answer)))
print(normalize_text(r"$\dfrac{14}{3.}$"))
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))
print(normalize_text("4/3"))

(14)/(3)
(14)/(3.)
(14)/(3)
4/3


#### 3.7. Verifying mathematical equivalence

In [65]:
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError
from tokenize import TokenError
from sympy.polys.polyerrors import PolynomialError

def sympy_parser(expr):
    # to avoid crashing on long garbage responses
    if expr is None or len(expr) > 2000:
        return None
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # standard transformation
                *spp.standard_transformations,
                # allow omitted multiplication symbols
                spp.implicit_multiplication_application,
            ),
            # evaluate during parsing so simple constrants simplify
            evaluate=True,
        )
    except (SympifyError, SyntaxError, TypeError, AttributeError, IndexError, TokenError, ValueError, PolynomialError):
        return None

In [69]:
sympy_parser(
    normalize_text(
        extract_final_candidate(
            model_answer
        )
    )
)

14/3

#### 3.8. Implementing the equality check

In [72]:
from sympy import simplify

def equality_check(expr_gtruth, expr_pred):
    # first, check if two are exactly same 
    if expr_gtruth == expr_pred:
        return True

    # parse both into Sympy objects (return None if parsing fails)
    gtruth, pred = sympy_parser(expr_gtruth), sympy_parser(expr_pred)

    # if both were parsed, try symbolic comparison
    if gtruth is not None and pred is not None:
        try:
            # if difference is 0, they are equivalent
            return simplify(gtruth - pred) == 0
        except (SympifyError, TypeError):
            pass

    return False

print(equality_check(
    normalize_text("13/4."),
    normalize_text(r"(13)/(4)")
))

print(equality_check(
    normalize_text("0.5"),
    normalize_text(r"(1)/(2)")
))

print(equality_check(
    normalize_text("14/3"),
    normalize_text("15/3")
))

# code is too simple
print(equality_check(
    normalize_text("(14/3, 2/3)"),
    normalize_text("(14/3, 4/6)")
))

True
True
False
False


#### 3.9. Building and testing the answer grader

In [87]:
def split_into_parts(text):
    result = [text]

    if text:
        # check if text tools like a tuple or list, e.g. `(a, b)` or `[a, b]`
        if (
            len(text) >= 2
            and text[0] in "(["
            and text[-1] in ")]"
            and "," in text[1:-1]
        ):
            # split on commas inside brackets and strip whitespaces
            items = [p.strip() for p in text[1:-1].split(",")]
            if all(items):
                result = items

    else:
        # if text is empty, return an empty list
        result =[]

    return result

split_into_parts("(14/3, 2/3)")

['14/3', '2/3']

In [88]:
def grade_answer(pred_text, gt_text):
    # default outcome if checks fail
    result = False

    # only continue if both inputs are non-empty
    if pred_text is not None and gt_text is not None:
        # break texts into comparable parts
        gt_parts = split_into_parts(normalize_text(gt_text))
        pred_parts = split_into_parts(normalize_text(pred_text))

        # ensure both side have same number of valid parts
        if (gt_parts and pred_parts
            and len(gt_parts) == len(pred_parts)):
            # check each part individually
            result = all(
                equality_check(gt, pred)
                for gt, pred in zip(gt_parts, pred_parts)
            )

    # True only if all checks passed
    return result

In [89]:
# Define test cases: (name, prediction, ground truth, expected result)
tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
        ("check_16", r"2²", "2**2", True),
    ]


def run_demos_table(tests):
    header = ("Test", "Expect", "Got", "Status")
    rows = []
    for name, pred, gtruth, expect in tests:
        got = grade_answer(pred, gtruth)  # Run equality check
        status = "PASS" if got == expect else "FAIL"
        rows.append((name, str(expect), str(got), status))

    data = [header] + rows
    
    # Compute max width for each column to align table nicely
    col_widths = [
        max(len(row[i]) for row in data)
        for i in range(len(header))
    ]

    # Print table row by row
    for row in data:
        line = " | ".join(
            row[i].ljust(col_widths[i])
            for i in range(len(header))
        )
        print(line)

    # Print summary of passed tests
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")

run_demos_table(tests)

Test     | Expect | Got   | Status
check_1  | True   | True  | PASS  
check_2  | True   | True  | PASS  
check_3  | True   | True  | PASS  
check_4  | True   | True  | PASS  
check_5  | True   | True  | PASS  
check_6  | False  | False | PASS  
check_7  | False  | False | PASS  
check_8  | True   | True  | PASS  
check_9  | False  | False | PASS  
check_10 | True   | True  | PASS  
check_11 | True   | True  | PASS  
check_12 | False  | False | PASS  
check_13 | True   | True  | PASS  
check_14 | True   | True  | PASS  
check_15 | True   | True  | PASS  
check_16 | True   | True  | PASS  

Passed 16/16


#### 3.10. Loading the evaluation dataset (MATH-500)

In [96]:

import json
import requests
from pathlib import Path

def load_math500_test(local_path="math500_test.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://huggingface.co/datasets/shubhamisok/"
        "math-500/resolve/main/math500_test.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

    return data


math_data = load_math500_test()

print("Number of entries:", len(math_data))

Number of entries: 500


#### 3.11. Evaluating the model

In [105]:
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template

prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)
prompt_fmt = render_prompt(prompt)
print(prompt_fmt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Problem:
If $a+b=3$ and $ab=\tfrac{13}{6}$, what is the value of $a^2+b^2$?

Answer:


In [106]:
generate_text = generate_text_stream_concat(
    model, tokenizer, prompt_fmt, device, max_new_tokens=2048, verbose=True
)

 \boxed{10}

In [109]:
def mini_eval_demo(model, tokenizer, device):
    ex = {  # Test example with "problem" and "answer" fields
        "problem": "Compute 1/2 + 1/6.",
        "answer": "2/3"
    }
    # apply prompt template
    prompt = render_prompt(ex["problem"])

    # generate reponse
    gen_text = generate_text_stream_concat(
        model, tokenizer, prompt, device,
        max_new_tokens=64,
    )

    # extract and normalize answer
    pred_answer = extract_final_candidate(gen_text)

    # grade answer
    is_correct = grade_answer(
        pred_answer, ex["answer"]
    )
    print(f"Device: {device}")
    print(f"Prediction: {pred_answer}")
    print(f"Ground truth: {ex['answer']}")
    print(f"Correct: {is_correct}")

mini_eval_demo(model, tokenizer, device)

Device: cuda
Prediction: 1/3
Ground truth: 2/3
Correct: False


In [110]:
import time

# Helper function to calculate remaining time
def eta_progress_message(
    processed,
    total,
    start_time,
    show_eta=False,
    label="Progress",
):
    progress = f"{label}: {processed}/{total}"
    pad_width = len(f"{label}: {total}/{total} | ETA: 00h 00m 00s")
    if not show_eta or processed <= 0:
        return progress.ljust(pad_width)

    elapsed = time.time() - start_time
    if elapsed <= 0:
        return progress.ljust(pad_width)

    remaining = max(total - processed, 0)

    if processed:
        avg_time = elapsed / processed
        eta_seconds = avg_time * remaining
    else:
        eta_seconds = 0

    eta_seconds = max(int(round(eta_seconds)), 0)
    minutes, rem_seconds = divmod(eta_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        eta = f"{hours}h {minutes:02d}m {rem_seconds:02d}s"
    elif minutes:
        eta = f"{minutes:02d}m {rem_seconds:02d}s"
    else:
        eta = f"{rem_seconds:02d}s"

    message = f"{progress} | ETA: {eta}"
    return message.ljust(pad_width)

In [ ]:
def evaluate_math500_stream(
        model, tokenizer, device, math_data, out_path=None, max_new_tokens=512, verbose=False
):
    if out_path is None:
        # make filename compatible
        dev_name = str(device).replace(":", "-")
        out_path = f"math500-{dev_name}.jsonl"

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    # save results to file
    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            # apply prompt template
            prompt = render_prompt(row['problem'])
            # generate response
            gen_text = generate_text_stream_concat(
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            # extract and normalize answer
            extracted = extract_final_candidate(gen_text)

            # grade answer
            is_correct = grade_answer(extracted, row["answer"])
            num_correct += is_correct

            # record to be saved
            record = {
                "index": i,
                "problem": row["problem"],
                "answer": row["answer"],
                "generated": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }

            # write to file
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            # progress  
            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500"
            )
            print(progress_msg, end="\r", flush=True)

            # print responses during generation process
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    second_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Time elapsed: {second_elapsed:.1f}s")
    print(f"Output saved to: {out_path}")

    return num_correct, num_examples, acc

In [113]:
print("Model: ", WHICH_MODEL)
print("Device: ", device)
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, math_data=math_data[:10], verbose=False
)

Model:  base
Device:  cuda
MATH-500: 10/10 | ETA: 00s        
Accuracy: 30.0% (3/10)
Time elapsed: 21.2s
Output saved to: math500-cuda.jsonl
